# Capítulo 4 — Inferência Local: vLLM, Batching e Streaming

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap04_inferencia_local.ipynb)

**Fontes originais:**
- [orca3/llm-model-serving — ch02/ch2_Run_LLM_With_vLLM.ipynb](https://github.com/orca3/llm-model-serving/tree/main/ch02)
- [orca3/llm-model-serving — ch02/ch2_Batching.ipynb](https://github.com/orca3/llm-model-serving/tree/main/ch02)
- [orca3/llm-model-serving — ch02/ch2_Streaming.ipynb](https://github.com/orca3/llm-model-serving/tree/main/ch02)

Adaptado e comentado por **Allan Eric Jepsen**.

Créditos aos autores originais do repositório [orca3/llm-model-serving](https://github.com/orca3/llm-model-serving).

## Configuração do Ambiente

Instalamos vLLM (motor de inferência otimizado), transformers (biblioteca base do HuggingFace) e tiktoken (tokenizador rápido).

In [ ]:
!pip install --quiet vllm transformers tiktoken

## Função utilitária para liberar memória GPU

Essencial para alternar entre diferentes configurações de modelo sem reiniciar o runtime.

In [ ]:
import torch
import gc
import time

def free_gpu(model):
    """Descarrega o modelo e libera a memória da GPU."""
    if model:
        del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    gc.collect()

free_gpu(None)

---
## 4.1 — Inferência com vLLM

O vLLM é um motor de inferência de alto desempenho que implementa **PagedAttention**, eliminando desperdício de memória no KV cache e permitindo throughput significativamente maior que o HuggingFace padrão.

Vamos comparar a latência do vLLM vs HuggingFace para o mesmo modelo e prompt.

### Definição do prompt

Usamos um prompt longo para simular um cenário realista de geração de texto.

In [ ]:
# Define o prompt de teste
prompt = """You are an expert AI historian writing a detailed chapter for a book titled "The Evolution of Human-AI Collaboration."

Begin by summarizing the early stages of artificial intelligence in the 1950s, touching on symbolic logic and rule-based systems. Then transition into the rise of machine learning, particularly deep learning in the 2010s.

Afterward, describe how large language models like GPT transformed human-computer interaction, enabling applications in education, creative writing, customer support, and software development.

Finally, reflect on the societal and ethical implications of AI, such as misinformation, bias, and the alignment problem.

Write in a formal tone, with rich detail and examples in each era."""

### Geração com vLLM

Carregamos o modelo Qwen2.5-0.5B com vLLM em float16 e medimos o tempo de inferência. Os `SamplingParams` controlam a estratégia de amostragem (temperatura, top_p, máximo de tokens).

In [ ]:
import time
from vllm import LLM, SamplingParams

model_name = "Qwen/Qwen2.5-0.5B"

# Carrega o modelo com vLLM
llm = LLM(model=model_name, dtype="float16")

# Parâmetros de amostragem
sampling_params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=128)

# Mede o tempo de geração
start_time = time.time()
outputs = llm.generate([prompt], sampling_params)
end_time = time.time()

# Imprime resultados
for output in outputs:
    print(f"Texto gerado: {output}")
    print(f"Tempo total: {end_time - start_time:.2f} segundos")

free_gpu(llm)

### Geração com HuggingFace (transformers)

Agora executamos o mesmo prompt com o pipeline padrão do HuggingFace para comparar. O HuggingFace é mais simples de usar, mas não é otimizado para throughput em produção.

In [ ]:
# --- Serviço Básico com HuggingFace (transformers) ---
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

start_time_basic = time.time()

# Carrega modelo e tokenizador
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", device_map="auto", trust_remote_code=True)

# Cria o pipeline de geração
generator = pipeline('text-generation', model=model, tokenizer=tokenizer)

outputs_basic = generator(prompt, max_length=128, temperature=0.8, top_p=0.95)
end_time_basic = time.time()

print("\n---- Resultados com HuggingFace ----")
for output in outputs_basic:
    print(f"Texto gerado: {output['generated_text']}")
    print(f"Tempo total: {end_time_basic - start_time_basic:.2f} segundos")

print(f"\nDiferença de latência: {(end_time_basic - start_time_basic) - (end_time - start_time):.2f} segundos")

free_gpu(generator)

---
## 4.2 — Batching: Processando Múltiplas Requisições em Paralelo

O **batching** é uma das técnicas mais importantes para aumentar o throughput de um servidor de LLM. Em vez de processar uma requisição por vez, o vLLM pode processar várias sequências simultaneamente, aproveitando o paralelismo da GPU.

Vamos comparar:
1. Processar 4 prompts **em lote** (batch)
2. Processar 4 prompts **um por um** (sequencial)

In [ ]:
from vllm import LLM, SamplingParams

# Carrega o modelo com vLLM
llm = LLM(
    model="Qwen/Qwen2.5-0.5B",
    dtype="float16",
    trust_remote_code=True,
    max_model_len=2048
)

### Comparação: Batch vs Sequencial

Enviamos 4 prompts diferentes e medimos o tempo total em cada abordagem. O batching deve ser significativamente mais rápido porque o vLLM usa **continuous batching** — novas sequências podem entrar no lote assim que outras terminam.

In [ ]:
import torch
import gc
import time
from vllm import LLM, SamplingParams
from transformers import pipeline

# Prompts para geração em lote — 4 sequências de entrada
prompts = [
    "What is the meaning of life?",
    "Write a short story about a robot learning to love.",
    "Explain quantum physics in simple terms.",
    "Translate 'Hello, world!' into Spanish."
]

sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=100
)

# Processa 4 sequências juntas em um lote
start_time = time.time()
vllm_outputs = llm.generate(prompts, sampling_params)
end_time = time.time()
vllm_time = end_time - start_time

print(f"\nTempo de geração vLLM para 4 prompts EM LOTE: {vllm_time:.4f} segundos")

# Processa prompts um por um (sequencial)
start_time = time.time()
for p in prompts:
    vllm_outputs = llm.generate([p], sampling_params)
end_time = time.time()
vllm_time = end_time - start_time

print(f"\nTempo de geração vLLM para 4 prompts UM POR UM: {vllm_time:.4f} segundos")

---
## 4.3 — Streaming: Retornando Tokens em Tempo Real

Em aplicações interativas (chatbots, assistentes), não queremos esperar o modelo gerar **todos** os tokens para começar a exibir a resposta. O **streaming** permite enviar cada token assim que ele é gerado.

Dois modos:
1. **Sem streaming**: retorna o resultado quando a geração completa
2. **Com streaming**: retorna cada token assim que disponível

### Configuração do motor assíncrono vLLM

O `AsyncLLMEngine` do vLLM permite geração assíncrona com streaming. Configuramos:
- `tensor_parallel_size`: número de GPUs para paralelismo
- `gpu_memory_utilization`: fração da memória GPU a utilizar
- `max_num_batched_tokens`: máximo de tokens processados por lote
- `max_num_seqs`: máximo de sequências simultâneas

In [ ]:
import asyncio
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.engine.async_llm_engine import AsyncLLMEngine
from vllm.sampling_params import SamplingParams

# Inicializa os argumentos do motor
engine_args = AsyncEngineArgs(
    model="Qwen/Qwen2.5-0.5B",
    dtype="float16",
    tensor_parallel_size=1,       # Número de GPUs
    gpu_memory_utilization=0.9,   # Utilização de memória GPU
    max_num_batched_tokens=32768, # Máximo de tokens por lote
    max_num_seqs=256,             # Máximo de sequências simultâneas
    disable_log_requests=True,    # Desabilita log de requisições
    disable_log_stats=True,       # Desabilita log de estatísticas
)

# Cria o motor assíncrono de streaming do vLLM
engine = AsyncLLMEngine.from_engine_args(engine_args)

### Função de geração com streaming

Esta função assíncrona envia o prompt ao motor e imprime cada token conforme ele é gerado, simulando a experiência de um chatbot em tempo real.

> **Nota:** Este código pode não funcionar diretamente no Jupyter devido ao event loop. Para execução completa, use o script [streaming.py](https://github.com/orca3/llm-model-serving/blob/main/ch02/streaming.py) no terminal.

In [ ]:
async def generate_text(prompt: str, max_tokens: int = 100, request_id="id"):
    """Gera texto de forma assíncrona com streaming de tokens."""
    try:
        # Define parâmetros de amostragem
        sampling_params = SamplingParams(
            temperature=0.0,
            max_tokens=max_tokens,
            stop=["\n"],  # Para ao encontrar quebra de linha
        )

        # Gera texto de forma assíncrona e em streaming
        results_generator = engine.generate(
            prompt=prompt,
            sampling_params=sampling_params,
            request_id=request_id
        )

        # Processa os resultados conforme chegam
        final_output = None
        async for request_output in results_generator:
            final_output = request_output
            # Imprime cada token conforme é gerado
            print("chunk")
            for output in request_output.outputs:
                print(output.text, end="", flush=True)
            print()
        print()

        print("\nGeração concluída com sucesso")
        return final_output

    except asyncio.CancelledError:
        print("\nGeração foi cancelada")
        return None
    finally:
        # Sempre faz limpeza
        try:
            await engine.abort(request_id)
        except:
            pass

In [ ]:
# Executa a geração com streaming
request_id = "any_id"
await generate_text(prompt, 10000, request_id)

> Se tiver problemas ao executar o código acima devido ao event loop do Jupyter, execute o exemplo [streaming.py](https://github.com/orca3/llm-model-serving/blob/main/ch02/streaming.py) no terminal.